# Iktos 3D Align API - Usage Guide

This notebook shows how to run the Iktos 3D Align API to align a list of SMILES on a reference ligand.

**Workflow overview:**
1. Initialize the API client
2. Check API quota
3. Prepare inputs (reference ligand + list of SMILES)
4. Submit a single prediction request
5. Monitor progress
6. Retrieve results into a dataframe

## 1. Setup

In [ ]:
import os

import pandas as pd
from rdkit import Chem

from align_3d_client import (
    Align3DClient,
    ReferenceLigandInput,
    LigandInput,
)

In [ ]:
API_URL = os.getenv("API_URL", "3D_ALIGN_API_URL")
API_KEY = os.getenv("API_KEY", "YOUR_API_KEY")

client = Align3DClient(API_URL, api_key=API_KEY)

## 2. Health Check & Quota

In [ ]:
health = client.health_check()
print(f"API Status: {health['status']}")

In [ ]:
quota = client.get_quota()
print(f"Quota: {quota.quota_used}/{quota.quota_max} used, {quota.quota_remaining} remaining")

## 3. Prepare Inputs

A prediction job requires:
- **One reference ligand** (`ReferenceLigandInput`): id + 3D molblock (SDF/MOL format)
- **A list of query ligands** (`LigandInput`): each with an id and a SMILES string

In [ ]:
REFERENCE_LIGAND_PATH = "path/to/reference_ligand.sdf"

reference_ligand = ReferenceLigandInput(
    id="reference_ligand",
    mol_block=Chem.MolToMolBlock(Chem.MolFromMolFile(REFERENCE_LIGAND_PATH)),
)
print(f"Reference ligand: {reference_ligand.id}")

In [ ]:
# List of query SMILES to align against the reference ligand
smiles_list = [
    "CCc1nn(CCO)c(CC)c1Oc1cc(C#N)cc(C#N)c1",
    "Cn1c(C#N)ccc1-c1ccc2c(c1)C(C)(C)OC(=S)N2",
    "Cc1csc(Nc2ncccc2Oc2ccccc2C#N)n1",
    # ... add as many SMILES as you like
]
ligands = [
    LigandInput(id=f"ligand_{i}", smiles=smi)
    for i, smi in enumerate(smiles_list)
]
print(f"Prepared {len(ligands)} ligands.")

## 4. Submit a Prediction Request


In [ ]:
response = client.create_request(
    reference_ligand=reference_ligand,
    ligands=ligands,
    batch_job_id="example_multi_smiles",
)

request_id = response["request_id"]
print(f"Request submitted. request_id: {request_id}")
print(f"Total ligands: {response['total_ligands']}")

## 5. Monitor Progress

You can either poll manually or use the built-in `wait_for_completion()` method. `wait_for_completion()` polls the request every `check_interval` seconds until all ligand jobs are in a terminal state. The optional `progress_callback` lets you print intermediate progress.

In [ ]:
# Manual status check
status = client.get_request_status(request_id)
summary = status.status_summary

print(f"Status: {status.status.value}")
print(f"Total:     {summary.total}")
print(f"Completed: {summary.completed} ({summary.completed_percent:.1f}%)")
print(f"Failed:    {summary.failed} ({summary.failed_percent:.1f}%)")
print(f"Pending:   {summary.pending + + summary.processing} ({(summary.processing_percent + summary.pending_percent):.1f}%)")

In [ ]:
def show_progress(request):
    s = request.status_summary
    if s is not None:
        print(
            f"  {s.completed}/{s.total} completed "
            f"({s.completed_percent:.1f}%)  |  failed: {s.failed}"
        )


final_status = client.wait_for_completion(
    request_id,
    check_interval=5,
    progress_callback=show_progress,
)

print(f"\nDone! Final status: {final_status.status.value}")

## 6. Retrieve Results

Each completed job returns:
- `pharmaco_score`
- `shape_score`
- `ligand_smiles`
- `ligand_pose_sdf` (predicted 3D pose)

In [ ]:
results = client.get_results(request_id)

rows = []
for ligand_id, res in results.items():
    row = {"ligand_id": ligand_id, "status": res["status"], "time_elapsed": res.get("time_elapsed")}
    if res["status"] == "COMPLETED":
        data = res["data"]
        row.update({
            "ligand_smiles": data.get("ligand_smiles"),
            "pharmaco_score": data.get("pharmaco_score"),
            "shape_score": data.get("shape_score"),
            "ligand_pose_sdf": data.get("ligand_pose_sdf"),
        })
    elif res["status"] == "FAILED":
        row["error"] = res.get("error")
    rows.append(row)

df = pd.DataFrame(rows)
df[[c for c in df.columns if c != "ligand_pose_sdf"]]

### Save predicted poses *(optional)*

In [ ]:
OUTPUT_DIR = "aligned_poses"
OUTPUT_FILE = "aligned_poses.sdf"
os.makedirs(OUTPUT_DIR, exist_ok=True)

sorted_df = df[df["ligand_pose_sdf"].notna()].copy()
sorted_df["_id_num"] = sorted_df["ligand_id"].str.extract(r"(\d+)$").astype(int)
sorted_df = sorted_df.sort_values("_id_num")

output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
with open(output_path, "w") as f:
    writer = Chem.SDWriter(f)
    for i, row in sorted_df.iterrows():
        mol = Chem.MolFromMolBlock(row["ligand_pose_sdf"], removeHs=False)
        if mol is None:
            continue
        mol.SetProp("_Name", str(row.get("ligand_id", f"ligand_{i}")))
        writer.write(mol)
    writer.close()

print(f"Saved {len(sorted_df)} poses to {output_path}")